## MariaDB setup

This notebook reads the dataset from the MariaDB table `sales_data` instead of a CSV file.

Install the Python packages in your notebook environment:
```bash
pip install pandas matplotlib sqlalchemy pymysql
```

Create/import your dataset into MariaDB as `sales_data` first. Configure the connection using environment variables `MARIADB_HOST`, `MARIADB_PORT`, `MARIADB_USER`, `MARIADB_PASSWORD`, and `MARIADB_DATABASE`. The notebook includes placeholder credentials as a fallback; replace them or set the environment variables before running it. Do not commit real passwords to version control.


In [15]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from sqlalchemy import create_engine
from sqlalchemy.engine import URL
from dotenv import load_dotenv

load_dotenv()

pd.set_option("display.float_format", "{:,.2f}".format)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

# MariaDB connection settings.
# Set these environment variables before launching Jupyter:
#   MARIADB_HOST, MARIADB_PORT, MARIADB_USER, MARIADB_PASSWORD, MARIADB_DATABASE
db_url = URL.create(
    drivername="mysql+pymysql",
    username=os.getenv("MARIADB_USER", "root"),
    password=os.getenv("MARIADB_PASSWORD", ""),
    host=os.getenv("MARIADB_HOST", "127.0.0.1"),
    port=int(os.getenv("MARIADB_PORT", "3306")),
    database=os.getenv("MARIADB_DATABASE", "restaurant"),
)
engine = create_engine(db_url)

# Corrected table name and date column (created_at instead of order_date)
df = pd.read_sql("SELECT * FROM orders", con=engine, parse_dates=["created_at"])
df["month_label"] = df["created_at"].dt.strftime("%b %Y")

print(df.shape)
display(df.head())

(6, 7)


,order_id,customer_id,staff_id,order_status,total_amount,created_at,month_label
0,1,1,4,Completed,24.49,2026-10-16 09:55:17,Oct 2026
1,2,2,5,Completed,19.50,2026-10-16 09:55:17,Oct 2026
2,3,3,4,Preparing,35.99,2026-10-16 09:55:17,Oct 2026
3,4,4,5,Pending,12.99,2026-10-16 09:55:17,Oct 2026
4,5,5,4,Completed,33.00,2026-10-16 09:55:17,Oct 2026


In [ ]:
print("Rows (line items):", len(df))
print("Distinct orders  :", df["order_id"].nunique())
print("Date range       :", df["order_date"].min().date(), "to", df["order_date"].max().date())
print("Missing values   :", int(df.isna().sum().sum()))

Rows (line items): 6
Distinct orders  : 6
Date range       : 2026-10-16 to 2026-10-16
Missing values   : 0


In [ ]:
total_revenue = df["subtotal"].sum()
total_orders  = df["order_id"].nunique()
items_sold    = df["quantity"].sum()
avg_order_val = total_revenue / total_orders

pd.DataFrame({"total_revenue": [total_revenue], "total_orders": [total_orders],
              "total_items_sold": [items_sold], "avg_order_value": [avg_order_val]})

In [ ]:
total_revenue = df["subtotal"].sum()
total_orders  = df["order_id"].nunique()
items_sold    = df["quantity"].sum()
avg_order_val = total_revenue / total_orders

pd.DataFrame({"total_revenue": [total_revenue], "total_orders": [total_orders],
              "total_items_sold": [items_sold], "avg_order_value": [avg_order_val]})

In [ ]:
q3 = (df.groupby(["item_id", "item_name"])
        .agg(units_sold=("quantity", "sum"), revenue=("subtotal", "sum"))
        .reset_index()
        .sort_values(["units_sold", "revenue"], ascending=False)
        .head(5))
q3

In [ ]:
q4 = (df.groupby(["year", "month_number", "month_label"])
        .agg(orders=("order_id", "nunique"), revenue=("subtotal", "sum"))
        .reset_index().sort_values(["year", "month_number"]))
q4

In [ ]:
q5 = (df.groupby("order_type")
        .agg(orders=("order_id", "nunique"), items_sold=("quantity", "sum"), revenue=("subtotal", "sum"))
        .sort_values("revenue", ascending=False).reset_index())
q5

In [ ]:
q6 = (df.assign(customer_name=df["first_name"] + " " + df["last_name"])
        .groupby(["customer_id", "customer_name"])
        .agg(total_orders=("order_id", "nunique"), total_spent=("subtotal", "sum"))
        .reset_index().sort_values("total_spent", ascending=False).head(5))
q6

In [ ]:
q7 = (df.assign(day_type=df["is_weekend"].map({1: "Weekend", 0: "Weekday"}))
        .groupby("day_type")
        .agg(orders=("order_id", "nunique"), revenue=("subtotal", "sum"), avg_line_value=("subtotal", "mean"))
        .sort_values("revenue", ascending=False).reset_index())
q7

In [ ]:
order_totals = df.groupby(["order_id", "order_type"])["subtotal"].sum().reset_index(name="order_total")
q8 = (order_totals.groupby("order_type")["order_total"]
        .agg(orders="count", avg_order_value="mean", smallest_order="min", largest_order="max")
        .sort_values("avg_order_value", ascending=False).reset_index())
q8

In [ ]:
by_item = (df.groupby(["category_name", "item_id", "item_name"])
             .agg(units_sold=("quantity", "sum"), revenue=("subtotal", "sum")).reset_index())
by_item["rank_in_category"] = by_item.groupby("category_name")["units_sold"].rank(method="min", ascending=False)
q9 = (by_item[by_item["rank_in_category"] == 1]
        .drop(columns="item_id").sort_values("category_name").reset_index(drop=True))
q9

In [ ]:
q10 = q4[["month_label", "revenue"]].copy()
q10["previous_month"] = q10["revenue"].shift(1)
q10["growth_pct"] = q10["revenue"].pct_change() * 100
q10.reset_index(drop=True)

In [ ]:
# ---- data for the three charts ----
trend = q4                                   # monthly revenue
top_items = q3.sort_values("units_sold")     # sorted so the biggest bar is on top

if "order_status" in df.columns:
    pie_col, pie_title = "order_status", "Order Status"
else:
    pie_col, pie_title = "order_type", "Orders by Type"
pie_data = df.drop_duplicates("order_id")[pie_col].value_counts()

# ---- layout ----
fig = plt.figure(figsize=(18, 10))
gs = fig.add_gridspec(2, 3, height_ratios=[0.6, 2], hspace=0.3, wspace=0.25)
fig.suptitle("Restaurant Sales & Reports Dashboard", fontsize=22, fontweight="bold", y=0.98)

# ---- 3 KPI cards ----
kpis = [("Total Revenue", f"₱{total_revenue:,.0f}", "#1f77b4"),
        ("Total Orders", f"{total_orders:,}", "#2ca02c"),
        ("Avg Order Value", f"₱{avg_order_val:,.2f}", "#ff7f0e")]
for i, (label, value, color) in enumerate(kpis):
    ax = fig.add_subplot(gs[0, i])
    ax.set_facecolor(color + "22")
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(True); s.set_edgecolor(color); s.set_linewidth(2)
    ax.text(0.5, 0.68, label, ha="center", va="center", fontsize=15, color="#444", transform=ax.transAxes)
    ax.text(0.5, 0.32, value, ha="center", va="center", fontsize=28, fontweight="bold", color=color, transform=ax.transAxes)

# ---- Sales trend line chart ----
ax1 = fig.add_subplot(gs[1, 0])
ax1.plot(trend["month_label"], trend["revenue"], marker="o", linewidth=2.5, color="#1f77b4")
ax1.fill_between(range(len(trend)), trend["revenue"], alpha=0.1, color="#1f77b4")
ax1.set_title("Sales Trend (Monthly Revenue)", fontsize=14, fontweight="bold")
ax1.set_ylabel("Revenue (₱)")
ax1.yaxis.set_major_formatter(mtick.StrMethodFormatter("₱{x:,.0f}"))
ax1.tick_params(axis="x", rotation=45)
ax1.grid(axis="y", alpha=0.3)

# ---- Top food items bar chart ----
ax2 = fig.add_subplot(gs[1, 1])
bars = ax2.barh(top_items["item_name"], top_items["units_sold"], color="#2ca02c")
ax2.bar_label(bars, padding=3)
ax2.set_title("Top 5 Food Items (Units Sold)", fontsize=14, fontweight="bold")
ax2.set_xlabel("Units sold")

# ---- Order status / type pie chart ----
ax3 = fig.add_subplot(gs[1, 2])
ax3.pie(pie_data.values, labels=pie_data.index, autopct="%1.1f%%", startangle=90,
        colors=["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"][:len(pie_data)],
        wedgeprops={"edgecolor": "white"})
ax3.set_title(pie_title, fontsize=14, fontweight="bold")

fig.savefig("dashboard.png", bbox_inches="tight")
plt.show()